In [ ]:
def detect_and_crop_images():
    """
    Detect textboxes in images and crop the detected areas.
    """

    # Check if model exists
    if not os.path.exists(MODEL_PATH):
        print(f"Error: Model file '{MODEL_PATH}' not found.")
        return

    # Check if input folder exists
    if not os.path.exists(INPUT_FOLDER):
        print(f"Error: Input folder '{INPUT_FOLDER}' does not exist.")
        return

    # Create output folder if it doesn't exist
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Load YOLO model
    print(f"Loading model from {MODEL_PATH}...")
    yolo_model = YOLO(MODEL_PATH)

    # Get all image files from input folder
    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"]
    image_files = []

    for ext in image_extensions:
        image_files.extend(Path(INPUT_FOLDER).glob(f"*{ext}"))
        image_files.extend(Path(INPUT_FOLDER).glob(f"*{ext.upper()}"))

    if not image_files:
        print(f"No image files found in {INPUT_FOLDER}")
        return

    print(f"Found {len(image_files)} images to process...")

    # Process each image
    for img_path in image_files:
        print(f"Processing: {img_path.name}")

        # Load image
        image = cv2.imread(str(img_path))
        if image is None:
            print(f"Could not load image: {img_path}")
            continue

        # Run detection
        results = yolo_model(image)

        # Process detections
        crop_count = 0
        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])
                confidence = box.conf[0]
                class_name = class_labels.get(cls, "Unknown")

                # Crop the detected area
                cropped = image[y1:y2, x1:x2]

                # Create filename for cropped image
                base_name = img_path.stem
                crop_filename = f"{base_name}_{class_name}_{crop_count:02d}.jpg"
                crop_path = os.path.join(OUTPUT_FOLDER, crop_filename)

                # Save cropped image
                cv2.imwrite(crop_path, cropped)
                print(f"  Saved crop: {crop_filename} (confidence: {confidence:.2f})")
                crop_count += 1

        if crop_count == 0:
            print(f"  No detections found in {img_path.name}")

    print(f"\nProcessing complete! Cropped images saved to: {OUTPUT_FOLDER}")

In [ ]:
def convert_to_grayscale():
    """
    Convert all images in the cropped_output folder to grayscale.
    """

    # Check if input folder exists
    if not os.path.exists(INPUT_FOLDER):
        print(f"Error: Input folder '{INPUT_FOLDER}' does not exist.")
        return

    # Create output folder if it doesn't exist
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Get all image files from input folder
    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"]
    image_files = []

    for ext in image_extensions:
        image_files.extend(Path(INPUT_FOLDER).glob(f"*{ext}"))
        image_files.extend(Path(INPUT_FOLDER).glob(f"*{ext.upper()}"))

    if not image_files:
        print(f"No image files found in {INPUT_FOLDER}")
        return

    print(f"Found {len(image_files)} images to convert to grayscale...")

    # Process each image
    for img_path in image_files:
        print(f"Converting: {img_path.name}")

        try:
            # Method 1: Using OpenCV
            image = cv2.imread(str(img_path))
            if image is not None:
                # Convert to grayscale
                gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

                # Create output filename
                base_name = img_path.stem
                output_filename = f"{base_name}_grayscale.jpg"
                output_path = os.path.join(OUTPUT_FOLDER, output_filename)

                # Save grayscale image
                cv2.imwrite(output_path, gray_image)
                print(f"  Saved: {output_filename}")

            else:
                # Method 2: Using PIL if OpenCV fails
                try:
                    with Image.open(img_path) as img:
                        # Convert to grayscale
                        gray_img = img.convert("L")

                        # Create output filename
                        base_name = img_path.stem
                        output_filename = f"{base_name}_grayscale.jpg"
                        output_path = os.path.join(OUTPUT_FOLDER, output_filename)

                        # Save grayscale image
                        gray_img.save(output_path, "JPEG", quality=95)
                        print(f"  Saved: {output_filename}")

                except Exception as e:
                    print(f"  Error processing {img_path.name}: {e}")

        except Exception as e:
            print(f"  Error processing {img_path.name}: {e}")

    print(f"\nConversion complete! Grayscale images saved to: {OUTPUT_FOLDER}")

In [ ]:
class BackgroundRemover:
    """
    Background removal using a canvas-style method (Otsu + morphology).
    Replaces background with solid white while preserving foreground strokes.
    """

    def load_image(self, image_path):
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")
        try:
            data = np.fromfile(image_path, dtype=np.uint8)
            img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        except Exception:
            img = None
        if img is None:
            raise ValueError(f"Could not read image: {image_path}")
        return img

    def remove_background_canvas_style(self, img):
        # Convert to grayscale if needed
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img.copy()

        # Slight blur for robustness
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        # Otsu threshold to separate foreground/background
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Invert so text/foreground = white, background = black
        mask = cv2.bitwise_not(thresh)

        # Clean mask
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # Compose on white background in grayscale
        white_bg = np.full_like(gray, 255, dtype=np.uint8)
        result = np.where(mask == 255, gray, white_bg).astype(np.uint8)
        return result

    def process_image_file(self, image_path, output_dir):
        img = self.load_image(image_path)
        result = self.remove_background_canvas_style(img)
        base_name = Path(image_path).stem
        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{base_name}.png")
        ok, buf = cv2.imencode(".png", result)
        if not ok:
            raise ValueError(f"Could not encode image: {image_path}")
        buf.tofile(out_path)
        print(f"Saved: {out_path}")

    def process_folder(self, input_dir, output_dir):
        if not os.path.isdir(input_dir):
            print(f"Input directory not found: {input_dir}")
            return 1

        files = [
            os.path.join(input_dir, f)
            for f in os.listdir(input_dir)
            if any(f.lower().endswith(ext) for ext in SUPPORTED_EXTS)
        ]

        if not files:
            print("No images found to process.")
            return 0

        print(f"Processing {len(files)} images from: {input_dir}")
        print(f"Output will be saved to: {output_dir}")

        for idx, fp in enumerate(files, 1):
            print(f"[{idx}/{len(files)}] {os.path.basename(fp)}")
            try:
                self.process_image_file(fp, output_dir)
            except Exception as exc:
                print(f"  Failed: {fp} -> {exc}")
        return 0